In [12]:
# pip install selenium pandas

<h1> Setting Up

In [13]:
import pandas as pd

import time
from datetime import datetime, timedelta

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

<h1> Scrape Booking.com's Data

<h2> Get Scraping's Time and Link

In [14]:
checkin_date = datetime.now() + timedelta(days=14)
checkout_date = checkin_date + timedelta(days=1)

checkin_str = checkin_date.strftime('%Y-%m-%d')
checkout_str = checkout_date.strftime('%Y-%m-%d')

# ur = "https://www.booking.com/searchresults.html?ss=London&group_adults=2&ltfd=1%3A5%3A11-2026%3A1%3A&no_rooms=1&group_children=0&selected_currency=USD"
url = (
    f"https://www.booking.com/searchresults.html?"
    f"ss=London&"
    f"group_adults=2&"
    f"no_rooms=1&"
    f"group_children=0&"
    f"checkin={checkin_str}&"
    f"checkout={checkout_str}&"
    f"selected_currency=USD"
)

print(f"Scraping for dates: {checkin_str} to {checkout_str}")

Scraping for dates: 2026-10-13 to 2026-10-14


<h2> Get Raw Data

In [15]:
# --- Setting Up Chrome Driver ---
driver = webdriver.Chrome()
driver.fullscreen_window()
driver.get(url)

time.sleep(10)

# Close Sign-In Button
try:
    close_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="Dismiss sign-in info."]'))
    )
    close_button.click()
except TimeoutException:
    pass

# Load More Data
LOAD_MORE_XPATH = "//button[contains(., 'Load more results')]"
max_loads = 100
max_scroll_steps = 30

for load_num in range(max_loads):
    load_more_btn = None

    for _ in range(max_scroll_steps):
        found = driver.find_elements(By.XPATH, LOAD_MORE_XPATH)  # returns [] instead of raising
        if found:
            load_more_btn = found[0]
            break
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(1)

    if load_more_btn is None:
        print("No more 'Load more results' button, stopping.")
        break

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", load_more_btn)
    # print(f"Clicked load more ({load_num + 1}/{max_loads})")

    time.sleep(3)
    
time.sleep(2)

# Get Raw Datas
hotel_cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')
print(f'Found total {len(hotel_cards)} property.')

No more 'Load more results' button, stopping.
Found total 643 property.


<h2> Parse Scraped Data

In [16]:
def get_data(card, css, attr="innerText"):
    try:
        return card.find_element(By.CSS_SELECTOR, css).get_attribute(attr)
    except NoSuchElementException:
        return None

data = []

for card in hotel_cards:
    data.append({
        'URL': get_data(card, '[data-testid="title-link"]', "href"),
        'Name': get_data(card, '[data-testid="title"]'),
        'Location': get_data(card, '[data-testid="address-link"]'),
        'Price': get_data(card, '[data-testid="price-and-discounted-price"]'),
        'Ratings': get_data(card, '[data-testid="review-score"]'),
        'Check_In_Date': checkin_date,
        'Scraped_At': datetime.now().isoformat(),
        'Search_URL': driver.current_url,
    })

<h2> Finishing

In [17]:
driver.quit()

# Convert to DataFrame
df = pd.DataFrame(data)
display(df.head())

,URL,Name,Location,Price,Ratings,Check_In_Date,Scraped_At,Search_URL
0,https://www.booking.com/hotel/gb/brit-hotels-e...,Brit Hotels Earls Court,"Kensington and Chelsea, London",US$269,Scored 8.3\n8.3\nVery Good\n734 reviews,2026-10-13 05:16:08.269463,2026-09-29T05:22:56.017755,https://www.booking.com/searchresults.html?ss=...
1,https://www.booking.com/hotel/gb/lancaster-gat...,Lancaster Gate Hotel,"Westminster Borough, London",US$303,"Scored 7.2\n7.2\nGood\n9,222 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.200365,https://www.booking.com/searchresults.html?ss=...
2,https://www.booking.com/hotel/gb/riu-plaza-lon...,Riu Plaza London Victoria,"Westminster Borough, London",US$553,"Scored 8.6\n8.6\nExcellent\n15,378 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.302497,https://www.booking.com/searchresults.html?ss=...
3,https://www.booking.com/hotel/gb/ibis-london-c...,ibis London City - Shoreditch,"Tower Hamlets, London",US$598,"Scored 8.1\n8.1\nVery Good\n6,553 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.393239,https://www.booking.com/searchresults.html?ss=...
4,https://www.booking.com/hotel/gb/sunborn-londo...,Sunborn London Yacht Hotel,"Newham, London",US$399,"Scored 8.8\n8.8\nExcellent\n6,053 reviews",2026-10-13 05:16:08.269463,2026-09-29T05:22:56.493018,https://www.booking.com/searchresults.html?ss=...


<h1> Export to CSV

In [18]:
# Export to CSV
df.to_csv('scraped_data.csv', index=False)